# Predicting the Source of a News Headline (Doc2Vec + ANN)

Given a news headline, predict which publisher wrote it.

**Steps:** load data → keep 6 publishers → clean text → train/test split → Doc2Vec (headline → 100 numbers) → ANN (numbers → publisher) → check accuracy

Dataset: [UCI News Aggregator (Kaggle)](https://www.kaggle.com/datasets/uciml/news-aggregator-dataset). Put `uci-news-aggregator.csv` next to this notebook.

In [1]:
import re
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Step 1: Load data and keep publishers with 2,000–3,000 headlines

In [2]:
df = pd.read_csv("uci-news-aggregator.csv")
news = df[["TITLE", "PUBLISHER"]].dropna()   # URL is dropped: it contains the publisher name

counts = news["PUBLISHER"].value_counts()
publishers = counts[(counts > 2000) & (counts < 3000)].index
news = news[news["PUBLISHER"].isin(publishers)].reset_index(drop=True)

print("Headlines:", len(news))
news["PUBLISHER"].value_counts()

Headlines: 13751


PUBLISHER
Huffington Post     2455
Businessweek        2395
Contactmusic.com    2334
Daily Mail          2254
NASDAQ              2228
Examiner.com        2085
Name: count, dtype: int64

## Step 2: Clean the text (lowercase, remove punctuation, split into words)

In [3]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9$%' ]", " ", text)
    return text.split()

news["words"] = news["TITLE"].apply(clean)
news[["TITLE", "words"]].head()

,TITLE,words
0,Plosser: Fed May Have to Accelerate Tapering Pace,"[plosser, fed, may, have, to, accelerate, tape..."
1,Forex - Pound drops to one-month lows against ...,"[forex, pound, drops, to, one, month, lows, ag..."
2,"Euro Anxieties Wane as Bunds Top Treasuries, S...","[euro, anxieties, wane, as, bunds, top, treasu..."
3,Noyer Says Strong Euro Creates Unwarranted Eco...,"[noyer, says, strong, euro, creates, unwarrant..."
4,EBay CEO Donahoe's Pay Drops by 53% to $13.8 M...,"[ebay, ceo, donahoe's, pay, drops, by, 53%, to..."


## Step 3: Split into train (80%) and test (20%)

In [4]:
train, test = train_test_split(news, test_size=0.2, stratify=news["PUBLISHER"], random_state=42)
print("Train:", len(train), " Test:", len(test))

Train: 11000  Test: 2751


## Step 4: Doc2Vec — turn every headline into 100 numbers

- `dm=0` → PV-DBOW mode (fast, good for short text)
- `dbow_words=1` → also learns word meanings at the same time, which gives better headline vectors
- Trained only on the training headlines

In [5]:
documents = [TaggedDocument(words, [i]) for i, words in enumerate(train["words"])]

d2v = Doc2Vec(dm=0, dbow_words=1, window=5, vector_size=100, min_count=2, epochs=40, workers=4, seed=42)
d2v.build_vocab(documents)
d2v.train(documents, total_examples=d2v.corpus_count, epochs=d2v.epochs)

X_train = np.array([d2v.infer_vector(w, epochs=40) for w in train["words"]])
X_test = np.array([d2v.infer_vector(w, epochs=40) for w in test["words"]])
print("Each headline is now", X_train.shape[1], "numbers")

Each headline is now 100 numbers


## Step 5: ANN — learn which numbers belong to which publisher

100 inputs → 128 neurons → 64 neurons → 6 outputs (one per publisher).
ReLU activation, Adam optimizer, early stopping to avoid overfitting.

In [6]:
scaler = StandardScaler().fit(X_train)
labels = LabelEncoder().fit(train["PUBLISHER"])

ann = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
                    alpha=1e-3, early_stopping=True, max_iter=300, random_state=42)
ann.fit(scaler.transform(X_train), labels.transform(train["PUBLISHER"]))
print("Training stopped after", ann.n_iter_, "epochs")

Training stopped after 19 epochs


## Step 6: Check accuracy on test headlines the model never saw

In [7]:
pred = labels.inverse_transform(ann.predict(scaler.transform(X_test)))

print("Accuracy:", round(accuracy_score(test["PUBLISHER"], pred), 3))
print(classification_report(test["PUBLISHER"], pred, digits=3))

Accuracy: 0.546
                  precision    recall  f1-score   support

    Businessweek      0.606     0.708     0.653       479
Contactmusic.com      0.534     0.531     0.533       467
      Daily Mail      0.508     0.426     0.463       451
    Examiner.com      0.470     0.305     0.370       417
 Huffington Post      0.398     0.501     0.444       491
          NASDAQ      0.755     0.783     0.769       446

        accuracy                          0.546      2751
       macro avg      0.545     0.542     0.539      2751
    weighted avg      0.544     0.546     0.540      2751



In [8]:
names = labels.classes_
pd.DataFrame(confusion_matrix(test["PUBLISHER"], pred, labels=names), index=names, columns=names)

,Businessweek,Contactmusic.com,Daily Mail,Examiner.com,Huffington Post,NASDAQ
Businessweek,339,5,11,11,44,69
Contactmusic.com,11,248,67,48,93,0
Daily Mail,42,74,192,29,106,8
Examiner.com,34,71,51,127,116,18
Huffington Post,53,66,56,52,246,18
NASDAQ,80,0,1,3,13,349


## Step 7: Try a new headline

In [9]:
def predict_source(headline):
    vector = scaler.transform([d2v.infer_vector(clean(headline), epochs=40)])
    probs = ann.predict_proba(vector)[0]
    top3 = np.argsort(probs)[::-1][:3]
    return [(names[i], round(float(probs[i]), 2)) for i in top3]

predict_source("Stocks fall as Fed hints at faster tapering")

[('Businessweek', 0.61), ('NASDAQ', 0.39), ('Huffington Post', 0.0)]